# Final Plotter

This notebook loads method CSV files from the project `outputs` folder and overlays them using `scienceplots` styling.

To make nearly overlapping curves easier to inspect, this version adds:
- method-specific colors and line styles
- markers on selected methods
- a second set of plots showing the difference from the `exact` result when available
- SVG export with filenames that follow the same parameter style as the CSV outputs


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science", "no-latex"])
    USING_SCIENCEPLOTS = True
except ImportError:
    USING_SCIENCEPLOTS = False
    print("scienceplots is not installed; using default matplotlib style.")

plt.rcParams["text.usetex"] = False
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
    }
)

PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

print(f"Using scienceplots: {USING_SCIENCEPLOTS}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figure directory: {FIGURE_DIR}")

In [ ]:
csv_paths = sorted(OUTPUT_DIR.glob("*.csv"))
csv_paths

## Optional File Selection

Leave `selected_csv_paths = csv_paths` to plot everything, or replace it with a smaller list to compare specific runs.


In [ ]:
selected_csv_paths = csv_paths

frames = []
for csv_path in selected_csv_paths:
    frame = pd.read_csv(csv_path)
    stem_parts = csv_path.stem.split("_")
    frame["method_name"] = stem_parts[0]
    frame["initial_state_type"] = stem_parts[1]
    frame["figure_metadata_suffix"] = "_".join(
        part
        for part in stem_parts[2:]
        if not part.startswith("numOfSamples") and not part.startswith("hilbertSize")
    )
    frames.append(frame)

if not frames:
    raise ValueError("No CSV files found in the outputs directory.")

shared_end_time = min(frame["time"].max() for frame in frames)
cropped_frames = [frame[frame["time"] <= shared_end_time].copy() for frame in frames]

initial_states = {frame["initial_state_type"].iloc[0] for frame in cropped_frames}
figure_metadata_suffixes = {frame["figure_metadata_suffix"].iloc[0] for frame in cropped_frames}

if len(initial_states) != 1:
    raise ValueError("Selected CSV files do not share the same initial_state_type.")
if len(figure_metadata_suffixes) != 1:
    raise ValueError("Selected CSV files do not share the same figure metadata suffix.")

shared_initial_state_type = next(iter(initial_states))
shared_metadata_suffix = next(iter(figure_metadata_suffixes))

print(f"Number of selected CSV files: {len(cropped_frames)}")
print(f"Shared end time after cropping: {shared_end_time}")
print(f"Shared initial state: {shared_initial_state_type}")
print(f"Shared metadata suffix: {shared_metadata_suffix}")

In [ ]:
style_map = {
    "exact": {"color": "black", "linestyle": "-", "linewidth": 2.3, "marker": None, "zorder": 5},
    "densityMatrix": {"color": "tab:blue", "linestyle": "--", "linewidth": 2.0, "marker": "^", "markevery": 400, "markersize": 3.6, "zorder": 4},
    "monteCarlo": {"color": "tab:orange", "linestyle": "-.", "linewidth": 1.9, "marker": "o", "markevery": 350, "markersize": 3.5, "zorder": 3},
    "positiveP": {"color": "tab:green", "linestyle": ":", "linewidth": 2.2, "marker": "s", "markevery": 450, "markersize": 3.2, "zorder": 2},
}

default_style = {"linewidth": 1.8, "linestyle": "-", "marker": None, "zorder": 1}

display_name_map = {
    "exact": "Exact",
    "densityMatrix": "Density Matrix",
    "monteCarlo": "Monte Carlo",
    "positiveP": "Positive-P",
}

plot_order = ["exact", "densityMatrix", "monteCarlo", "positiveP"]

def get_style(method_name):
    style = default_style.copy()
    style.update(style_map.get(method_name, {}))
    return style

def get_difference_style(method_name):
    style = get_style(method_name)
    style["marker"] = None
    style["linestyle"] = "-"
    style.pop("markevery", None)
    style.pop("markersize", None)
    return style

def build_figure_path(prefix):
    filename = f"{shared_initial_state_type}_{prefix}_{shared_metadata_suffix}.svg"
    return FIGURE_DIR / filename

def parse_metadata_value(prefix):
    for part in shared_metadata_suffix.split("_"):
        if part.startswith(prefix):
            raw_value = part[len(prefix):]
            # Keep backward compatibility with older p-style filenames.
            return raw_value.replace("p", ".")
    return None

def build_state_title_text():
    num_of_particles_text = parse_metadata_value("numOfParticles")
    gamma_text = parse_metadata_value("gamma")
    if shared_initial_state_type == "fock":
        state_text = rf"Fock state $|{num_of_particles_text}\rangle$"
    else:
        alpha_text = f"{float(num_of_particles_text) ** 0.5:g}"
        state_text = rf"Coherent state $|\alpha={alpha_text}\rangle$"
    return state_text, gamma_text

In [ ]:
state_title_text, gamma_text = build_state_title_text()

mean_fig, mean_ax = plt.subplots(figsize=(7.5, 5.2))

for ordered_method_name in plot_order:
    for frame in cropped_frames:
        method_name = frame["method_name"].iloc[0]
        if method_name != ordered_method_name:
            continue
        style = get_style(method_name)
        mean_ax.plot(
            frame["time"],
            frame["mean_particle_number"],
            label=display_name_map.get(method_name, method_name),
            **style,
        )

mean_ax.set_xlabel("Time")
mean_ax.set_ylabel("Mean Particle Number")
mean_ax.set_title(rf"Mean vs Time ({state_title_text}, $\gamma={gamma_text}$)")
mean_ax.legend()
mean_ax.grid(True, alpha=0.35)

mean_output_path = build_figure_path("mean")
mean_fig.savefig(mean_output_path, format="svg", bbox_inches="tight")
plt.show()

print(f"Saved mean plot to: {mean_output_path}")

In [ ]:
variance_fig, variance_ax = plt.subplots(figsize=(7.5, 5.2))

for ordered_method_name in plot_order:
    for frame in cropped_frames:
        method_name = frame["method_name"].iloc[0]
        if method_name != ordered_method_name:
            continue
        style = get_style(method_name)
        variance_ax.plot(
            frame["time"],
            frame["variance"],
            label=display_name_map.get(method_name, method_name),
            **style,
        )

variance_ax.set_xlabel("Time")
variance_ax.set_ylabel("Variance")
variance_ax.set_title(rf"Variance vs Time ({state_title_text}, $\gamma={gamma_text}$)")
variance_ax.legend()
variance_ax.grid(True, alpha=0.35)

variance_output_path = build_figure_path("variance")
variance_fig.savefig(variance_output_path, format="svg", bbox_inches="tight")
plt.show()

print(f"Saved variance plot to: {variance_output_path}")

## Difference From Exact

When curves overlap strongly, plotting the difference from the `exact` result is often the clearest way to see small deviations.


In [ ]:
exact_frame = None
for frame in cropped_frames:
    if frame["method_name"].iloc[0] == "exact":
        exact_frame = frame[["time", "mean_particle_number", "variance"]].reset_index(drop=True)
        break

if exact_frame is None:
    print("No exact dataset found, so the difference plots were skipped.")
else:
    mean_diff_fig, mean_diff_ax = plt.subplots(figsize=(7.5, 5.2))
    for ordered_method_name in plot_order:
        if ordered_method_name == "exact":
            continue
        for frame in cropped_frames:
            method_name = frame["method_name"].iloc[0]
            if method_name != ordered_method_name:
                continue
            mean_difference = frame["mean_particle_number"].to_numpy() - exact_frame["mean_particle_number"].to_numpy()
            mean_diff_ax.plot(
                frame["time"],
                mean_difference,
                label=display_name_map.get(method_name, method_name),
                **get_difference_style(method_name),
            )

    mean_diff_ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
    mean_diff_ax.set_xlabel("Time")
    mean_diff_ax.set_ylabel("Mean difference from exact")
    mean_diff_ax.set_title(rf"Mean Difference vs Time ({state_title_text}, $\gamma={gamma_text}$)")
    mean_diff_ax.legend()
    mean_diff_ax.grid(True, alpha=0.35)

    mean_diff_output_path = build_figure_path("meanDifference")
    mean_diff_fig.savefig(mean_diff_output_path, format="svg", bbox_inches="tight")
    plt.show()
    print(f"Saved mean difference plot to: {mean_diff_output_path}")

    variance_diff_fig, variance_diff_ax = plt.subplots(figsize=(7.5, 5.2))
    for ordered_method_name in plot_order:
        if ordered_method_name == "exact":
            continue
        for frame in cropped_frames:
            method_name = frame["method_name"].iloc[0]
            if method_name != ordered_method_name:
                continue
            variance_difference = frame["variance"].to_numpy() - exact_frame["variance"].to_numpy()
            variance_diff_ax.plot(
                frame["time"],
                variance_difference,
                label=display_name_map.get(method_name, method_name),
                **get_difference_style(method_name),
            )

    variance_diff_ax.axhline(0.0, color="black", linewidth=1.0, alpha=0.6)
    variance_diff_ax.set_xlabel("Time")
    variance_diff_ax.set_ylabel("Variance difference from exact")
    variance_diff_ax.set_title(rf"Variance Difference vs Time ({state_title_text}, $\gamma={gamma_text}$)")
    variance_diff_ax.legend()
    variance_diff_ax.grid(True, alpha=0.35)

    variance_diff_output_path = build_figure_path("varDifference")
    variance_diff_fig.savefig(variance_diff_output_path, format="svg", bbox_inches="tight")
    plt.show()
    print(f"Saved variance difference plot to: {variance_diff_output_path}")

## Notes

Our project uses one CSV per simulation run, with columns:
- `time`
- `mean_particle_number`
- `variance`

So unlike your older plotting script, we do not need separate files like `exact_fock_mean.csv` and `exact_fock_variance.csv`. Both are already stored in the same CSV.
